# 01 — BigEarthNet Dataset Exploration

This notebook explores the BigEarthNet-S2 and BigEarthNet-S1 datasets used to fine-tune SatQuery AI models.

## What you'll do
1. Load and inspect BigEarthNet patches
2. Visualise multi-band Sentinel-2 composites
3. Explore Sentinel-1 SAR imagery
4. Analyse label distributions
5. Generate text descriptions for CLIP training


In [ ]:
import sys
sys.path.insert(0, '../backend')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR = os.environ.get('BIGEARTHNET_DIR', '/data/BigEarthNet')
S2_ROOT = os.path.join(DATA_DIR, 'BigEarthNet-S2')

print(f'Data dir: {DATA_DIR}')
print(f'S2 root exists: {os.path.exists(S2_ROOT)}')

In [ ]:
from training.bigearthnet_dataset import BigEarthNetDataset, BIGEARTHNET_43_LABELS

# Load a small subset for exploration
ds = BigEarthNetDataset(
    data_dir=DATA_DIR,
    split='train',
    use_sar=True,
    max_samples=500,
)

print(f'Dataset size: {len(ds)}')
print(f'Number of labels: {len(BIGEARTHNET_43_LABELS)}')

In [ ]:
# ── Inspect a single sample ──────────────────────────────────────────────────
optical, sar, labels, description = ds[0]

print(f'Optical tensor shape: {optical.shape}')  # [12, 120, 120]
print(f'SAR tensor shape:     {sar.shape}')       # [2, 120, 120]
print(f'Labels shape:         {labels.shape}')    # [43]
print(f'Active labels:        {labels.sum().int().item()}')
print(f'Description:          {description}')

In [ ]:
# ── Visualise RGB composite + SAR ─────────────────────────────────────────────
def show_sample(ds, idx):
    optical, sar, labels, desc = ds[idx]
    label_names = [BIGEARTHNET_43_LABELS[i] for i, v in enumerate(labels) if v > 0.5]

    # De-normalise RGB bands (B04=idx3, B03=idx2, B02=idx1)
    S2_MEAN = np.array([340.76, 429.9, 614.21, 590.23, 950.68, 1792.53,
                        2075.11, 2218.94, 2266.46, 732.95, 1648.9, 1049.44])
    S2_STD  = np.array([554.99, 572.41, 582.87, 675.88, 729.89, 1096.01,
                        1273.45, 1365.45, 1356.13, 1108.06, 1258.32, 1008.28])

    rgb_idx = [3, 2, 1]
    rgb = optical[rgb_idx].numpy()  # [3, 120, 120]
    for i, bi in enumerate(rgb_idx):
        rgb[i] = rgb[i] * S2_STD[bi] + S2_MEAN[bi]
    rgb = rgb.transpose(1, 2, 0)
    p2, p98 = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)

    # SAR VV
    vv = sar[0].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.patch.set_facecolor('#0a1020')

    axes[0].imshow(rgb)
    axes[0].set_title('RGB (B04-B03-B02)', color='white', fontsize=10)
    axes[0].axis('off')

    # NDVI proxy
    nir = optical[6].numpy() * S2_STD[6] + S2_MEAN[6]
    red = optical[3].numpy() * S2_STD[3] + S2_MEAN[3]
    ndvi = (nir - red) / (nir + red + 1e-6)
    axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    axes[1].set_title('NDVI (B08-B04)/(B08+B04)', color='white', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(vv, cmap='gray')
    axes[2].set_title('SAR VV (normalised)', color='white', fontsize=10)
    axes[2].axis('off')

    for ax in axes:
        ax.set_facecolor('#050811')

    fig.suptitle(
        f'Sample {idx} | Labels: {", ".join(label_names[:3])}{"..." if len(label_names) > 3 else ""}\n{desc}',
        color='white', fontsize=9, y=0.02
    )
    plt.tight_layout()
    plt.show()

show_sample(ds, 0)
show_sample(ds, 5)

In [ ]:
# ── Label distribution ────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=64, num_workers=0)
label_counts = Counter()

for _, _, labels_batch, _ in loader:
    for i, lbl in enumerate(BIGEARTHNET_43_LABELS):
        label_counts[lbl] += int(labels_batch[:, i].sum().item())

top_labels = sorted(label_counts.items(), key=lambda x: -x[1])[:20]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0a1020')
ax.set_facecolor('#0f1a30')

names, counts = zip(*top_labels)
bars = ax.barh(range(len(names)), counts, color='#3aabff', alpha=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9, color='white')
ax.set_xlabel('Sample count', color='white')
ax.set_title('Top-20 BigEarthNet Label Frequencies', color='white', fontsize=11)
ax.tick_params(axis='x', colors='white')
ax.spines[['top', 'right']].set_visible(False)
for spine in ax.spines.values():
    spine.set_color('#334155')

plt.tight_layout()
plt.show()

In [ ]:
# ── Text descriptions for CLIP ────────────────────────────────────────────────
from training.bigearthnet_dataset import BigEarthNetDataset

sample_descriptions = [
    BigEarthNetDataset.generate_text_description(labels[:n])
    for labels, n in [
        (['Mixed forest', 'Water bodies', 'Pastures'], 3),
        (['Continuous urban fabric'], 1),
        (['Arable land', 'Non-irrigated arable land'], 2),
        (['Sea and ocean', 'Coastal lagoons', 'Beaches, dunes, sands', 'Intertidal flats'], 4),
    ]
]

print('Generated CLIP text descriptions:')
for i, d in enumerate(sample_descriptions, 1):
    print(f'  {i}. {d}')